In [ ]:
from dotenv import load_dotenv
from typing import TypedDict
import json
import requests
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.tools import tool

load_dotenv()

class Artist(TypedDict):
    id: int
    name: str
    link: str
    picture: str

@tool
def search_song(query: str) -> str:
    """Cherche un titre/musique sur Deezer. Ex: 'Daft Punk Get Lucky'. Retourne titre, artiste, album, preview 30s."""
    r = requests.get("https://api.deezer.com/search", params={"q": query}, timeout=10)
    r.raise_for_status()
    out = []
    for t in r.json().get("data", [])[:5]:
        out.append({
            "titre": t["title"],
            "artiste": t["artist"]["name"],
            "album": t["album"]["title"],
            "duree_sec": t["duration"],
            "preview_mp3": t["preview"],
            "lien": t["link"],
            "id": t["id"],
        })
    if not out:
        return f"Aucun titre trouvé pour '{query}'."
    return json.dumps(out, ensure_ascii=False)

@tool
def search_artist(nom: str) -> str:
    """Cherche un artiste sur Deezer. Ex: 'Daft Punk'. Retourne id, nom, fans, nb_albums."""
    r = requests.get("https://api.deezer.com/search/artist", params={"q": nom}, timeout=10)
    r.raise_for_status()
    out = []
    for a in r.json().get("data", [])[:5]:
        out.append({
            "nom": a["name"],
            "id": a["id"],
            "fans": a["nb_fan"],
            "nb_albums": a["nb_album"],
            "lien": a["link"],
            "picture": a["picture_medium"],
        })
    if not out:
        return f"Artiste '{nom}' introuvable."
    return json.dumps(out, ensure_ascii=False)

@tool
def top_titres_artist(nom_ou_id: str) -> str:
    """Donne le top titres d'un artiste Deezer. Input: nom ('Daft Punk') ou id ('27'). Retourne top 10 titres."""
    # 1. résoudre l'id si on a un nom
    artist_id = nom_ou_id
    if not str(nom_ou_id).isdigit():
        s = requests.get("https://api.deezer.com/search/artist", params={"q": nom_ou_id}, timeout=10).json()
        if not s.get("data"):
            return f"Artiste '{nom_ou_id}' introuvable."
        artist_id = s["data"][0]["id"]
    # 2. top titres
    r = requests.get(f"https://api.deezer.com/artist/{artist_id}/top", params={"limit": 10}, timeout=10)
    r.raise_for_status()
    out = []
    for i, t in enumerate(r.json().get("data", []), 1):
        out.append({
            "rang": i,
            "titre": t["title"],
            "artiste": t["artist"]["name"],
            "album": t["album"]["title"],
            "preview_mp3": t["preview"],
            "lien": t["link"],
        })
    if not out:
        return f"Aucun top titre trouvé pour '{nom_ou_id}'."
    return json.dumps(out, ensure_ascii=False)

# --- Agent ---
model = ChatGroq(model="openai/gpt-oss-120b")
agent = create_agent(model, tools=[search_song, search_artist, top_titres_artist])

result = agent.invoke({"messages": [("user", "C'est qui Daft Punk ? Donne son top 3 titres et cherche la chanson Get Lucky")]})
print(result["messages"][-1].content)

